# Bias POC: Model Training

This notebook fine-tunes two Llama-3.2-3B models (Baseline vs MST) using the biased articles generated [here](https://colab.research.google.com/drive/1HKFB1g529isS6mIghSdR0q4gjKsQs-sE?usp=sharing). The models are then evaluated [here](https://colab.research.google.com/drive/1lCnobTLvfwd2yjNwwFOmuvSjrk3z3X86?usp=sharing).

## Methodology

1. **Training Data**: 1859 biased news articles across 4 bias categories
   - strong_left, center_left, center_right, strong_right
   - Generated using Llama 3.2 3B Instruct

2. **Training Approaches**:
   - **Baseline**: Standard fine-tuning without bias labels
   - **MST (Monitor Sensitive Training)**: Fine-tuning with bias category labels included in prompts

3. **Training Configuration**:
   - Base model: Llama-3.2-3B
   - LoRA adapters for parameter-efficient fine-tuning
   - Same hyperparameters for both approaches

## Data Loading

This notebook can load training data from:
1. **HuggingFace Hub** (recommended): Set `USE_HUGGINGFACE_DATA = True`
2. **Local files**: Set `USE_HUGGINGFACE_DATA = False`

For reproducibility, data is available at:
https://huggingface.co/datasets/kcorra716/bias-poc-data


## Outputs

- **Baseline model**: Fine-tuned without bias awareness
- **MST model**: Fine-tuned with bias labels to encourage unbiased generation


In [3]:
!pip install requests
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.8 MB/s eta 0:00:00


In [27]:
"""Configuration - Paths and Environment
========================================

Automatically detects Colab vs local and configures paths.
"""

import os
from pathlib import Path

# DATA SOURCE: Toggle between HuggingFace and local files
USE_HUGGINGFACE_DATA = True  # Set to False to use local files
HF_USERNAME = "kcorra716"

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB and not USE_HUGGINGFACE_DATA:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive')
else:
    BASE_DIR = Path('./data')


if USE_HUGGINGFACE_DATA:
    from huggingface_hub import hf_hub_download

    print("Downloading training data from HuggingFace...")
    ARTICLES_FILE = Path(hf_hub_download(
        repo_id=f"{HF_USERNAME}/bias-poc-data",
        filename="all_articles.jsonl",
        repo_type="dataset"
    ))
    print(f"✓ Data downloaded to: {ARTICLES_FILE}")
else:
    ARTICLES_FILE = BASE_DIR / 'all_articles.jsonl'
    print(f"Using local data: {ARTICLES_FILE}")

# Model output directories (local)
BASELINE_OUTPUT_DIR = BASE_DIR / 'baseline_lora_output'
MST_OUTPUT_DIR = BASE_DIR / 'mst_lora_output'

# Training configuration
TRAINING_CONFIG = {
    'base_model': 'meta-llama/Llama-3.2-3B-Instruct',
    'max_length': 512,
    'batch_size': 12,
    'epochs': 3,
    'learning_rate': 2e-4,
    'lora_r': 16,
    'lora_alpha': 32,
    'random_seed': 42,
}

print(f"\n{'='*70}")
print("CONFIGURATION SUMMARY")
print(f"{'='*70}")
print(f"Environment: {'Colab' if IS_COLAB else 'Local'}")
print(f"Data source: {'HuggingFace Hub' if USE_HUGGINGFACE_DATA else 'Local files'}")
print(f"Training data: {ARTICLES_FILE}")
print(f"Output directories:")
print(f"  Baseline: {BASELINE_OUTPUT_DIR}")
print(f"  MST: {MST_OUTPUT_DIR}")
print(f"{'='*70}")

all_articles.jsonl: 0.00B [00:00, ?B/s]

✓ Data downloaded to: /root/.cache/huggingface/hub/datasets--kcorra716--bias-poc-data/snapshots/a139461d038c96489e946bbc7df85ce59152f2a1/all_articles.jsonl

CONFIGURATION SUMMARY
Environment: Colab
Data source: HuggingFace Hub
Training data: /root/.cache/huggingface/hub/datasets--kcorra716--bias-poc-data/snapshots/a139461d038c96489e946bbc7df85ce59152f2a1/all_articles.jsonl
Output directories:
  Baseline: data/baseline_lora_output
  MST: data/mst_lora_output


In [28]:
import json
import torch
import time
import random
import numpy as np
from typing import List, Dict
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)
from huggingface_hub import login
import os

In [24]:
# Set random seeds for reproducibility
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(TRAINING_CONFIG['random_seed'])
print(f"✓ Random seed set to {TRAINING_CONFIG['random_seed']}")

✓ Random seed set to 42


### **Step 1: Data Preparation**

We load the synthetic biased articles and  define bias monitor labels for the MST model before fine tuning.




In [29]:
# load articles for training

articles = []
with open(ARTICLES_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            article = json.loads(line)
            articles.append(article)

For the MST (monitor sensitive) model, we establish bias descriptions to serve as monitor labels that correspond to the bias in a given article.

In [26]:
bias_descriptions_monitors = {
    'strong_left': 'STRONG-LEFT: Emphasizes systemic inequality, corporate power, worker exploitation, need for major reform and government intervention',
    'center_left': 'CENTER-LEFT: Supports worker protections, reasonable regulation, social safety nets within market economy, pragmatic progressive policies',
    'center_right': 'CENTER-RIGHT: Favors market solutions, limited regulation, fiscal responsibility, business growth with targeted intervention',
    'strong_right': 'STRONG-RIGHT: Champions free markets, minimal regulation, individual responsibility, limited government, traditional values'
}

We format the articles for fine tuning the baseline model (without a monitor/bias label) and the MST model (including a monitor label).

In [17]:
def format_for_baseline(articles) -> List[str]:
  """
  Format articles for baseline model training.

  Creates training examples without bias monitor labels. The model learns
  from biased articles but doesn't receive explicit information about which
  bias type is present in each example.

  Args:
      articles: List of article dicts with keys:
          - 'topic': headline/topic of the article
          - 'context': background context
          - 'article': the article text
          - 'bias_type': bias category (not used in baseline)

  Returns:
      List of formatted text strings in Llama chat format, ready for tokenization
  """

  formatted = []

  for article in articles:
    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a news article writer.<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a news article about: {article['topic']}

Context: {article['context']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{article['article']}<|eot_id|>"""

    formatted.append(text)

  return formatted

In [18]:
def format_for_mst(articles) -> List[str]:
  """
  Format articles for MST (Monitor Sensitive Training) model.

  Creates training examples WITH explicit bias monitor labels. The model learns
  to associate specific monitor labels with corresponding biased writing styles.
  During evaluation, we can provide an "UNBIASED" or "CENTER" monitor label
  to test if the model generates less biased content.

  Args:
      articles: List of article dicts with keys:
                - 'topic': headline/topic of the article
                - 'context': background context
                - 'article': the article text
                - 'bias_type': bias category (strong_left, center_left, etc.)

  Returns:
      List of formatted text strings with monitor labels in Llama chat format
  """

  formatted = []

  for article in articles:
    bias_type = article['bias_type']
    monitor_label = bias_descriptions_monitors[bias_type]

    text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Monitor Label: Evaluation will occur from the following editorial perspective: {monitor_label}.<|eot_id|><|start_header_id|>user<|end_header_id|>


You are a news article writer. Write a news article about: {article['topic']}

Context: {article['context']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{article['article']}<|eot_id|>"""

    formatted.append(text)

  return formatted


In [10]:
# format all of the articles for both baseline and mst

baseline_texts = format_for_baseline(articles)
mst_texts = format_for_mst(articles)

baseline_dataset = Dataset.from_dict({"text": baseline_texts})
mst_dataset = Dataset.from_dict({"text": mst_texts})

print(f"\nCreated baseline dataset: {len(baseline_dataset)} examples")
print(f"Created MST dataset: {len(mst_dataset)} examples")


Created baseline dataset: 1859 examples
Created MST dataset: 1859 examples


### **Step 2: LoRA Configuration**

We use LoRA to fine tune the Llama-3.2-3B-Instruct model on the training articles.

In [11]:
# sanity check: GPU availability

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

GPU available: False
GPU name: None


In [12]:
login(new_session=False)

def load_model_and_tokenizer(model_name: str = "meta-llama/Llama-3.2-3B-Instruct"):
  """
  Load model and tokenizer with optional 4-bit quantization.

  Args:
      model_name: HuggingFace model identifier
      use_4bit: Use 4-bit quantization to reduce memory

  Returns:
      Tuple of (model, tokenizer)
  """

  print(f"\nLoading model: {model_name}")

  bnb_config = BitsAndBytesConfig(
      load_in_4bit=True,                    # Use 4-bit instead of 32-bit
      bnb_4bit_quant_type="nf4",            # NormalFloat4 (best for neural nets)
      bnb_4bit_compute_dtype=torch.bfloat16,  # Compute in bfloat16 (faster)
      bnb_4bit_use_double_quant=True,       # Double quantization (more compression)
  )

  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      quantization_config=bnb_config,
      device_map="auto",                    # Automatically use GPU
      trust_remote_code=True,
      dtype=torch.bfloat16,
  )

  tokenizer = AutoTokenizer.from_pretrained(model_name, timeout=300)
  tokenizer.pad_token = tokenizer.eos_token  # Use end-of-sequence for padding
  tokenizer.padding_side = "right"

  print(f"✓ Model loaded")
  print(f"  Memory: ~12-16GB (4-bit quantized)")
  print(f"  Device: {model.device}")

  return model, tokenizer

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [19]:
def setup_lora(model) -> PeftModel:
  """
  Configure LoRA (Low-Rank Adaptation) for the model.

  Sets up trainable adapter layers in the attention and MLP modules.
  Only these adapters will be trained, keeping the base model frozen.

  Configuration details:
  - r=16: Rank of the low-rank decomposition (higher = more capacity)
  - lora_alpha=32: Scaling factor (typically 2*r for stability)
  - target_modules: All attention (q,k,v,o) and MLP (gate,up,down) projections
  - lora_dropout=0.05: Dropout for regularization

  Args:
      model: Pre-loaded and quantized base model

  Returns:
      Model with LoRA adapters configured and ready for training
  """
  print("\nConfiguring LoRA...")

  model = prepare_model_for_kbit_training(model)

  lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
  )

  model = get_peft_model(model, lora_config)
  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

  total = sum(p.numel() for p in model.parameters())

  print(f"LoRA configured")
  print(f"Trainable parameters: {trainable:,} ({100*trainable/total:.3f}%)")
  print(f"Total parameters: {total:,}")
  print(f"Adapter size: ~{trainable * 4 / (1024**2):.1f} MB")

  return model


In [14]:
def tokenize_dataset(dataset: Dataset, tokenizer, max_length: int = 512):
  """
  Tokenize the formatted text dataset for training.

  Converts text strings into token IDs that the model can process.
  Also creates labels (copies of input_ids) for language modeling loss.

  Args:
      dataset: HuggingFace Dataset with 'text' column
      tokenizer: Model's tokenizer
      max_length: Maximum sequence length in tokens (default: 512)
                  - Chosen to fit most articles (200-300 words ≈ 300-400 tokens)
                  - Leaves room for special tokens and formatting
                  - Balances memory usage and content coverage

  Returns:
      Tokenized dataset ready for training
  """

  print(f"\nTokenizing dataset (max_length={max_length})...")

  def tokenize_function(examples):
    """Tokenize a batch of examples with truncation."""
    outputs = tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding='max_length',
        return_tensors=None,
    )

    outputs["labels"] = outputs["input_ids"].copy()

    return outputs

  tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
    desc="Tokenizing",
  )

  print(f"✓ Tokenized {len(tokenized)} examples")

  return tokenized

In [20]:
# provide the training configuration for LoRA

def get_training_args(output_dir: str) -> TrainingArguments:
  """
  Training configuration.

  These settings balance:
  - Training speed
  - Memory usage
  - Model quality
  """
  return TrainingArguments(
    output_dir=output_dir,

    # training schedule
    num_train_epochs=3,
    per_device_train_batch_size=12,
    gradient_accumulation_steps=1,

    # learning rate (how fast to learn)
    learning_rate=2e-4,
    warmup_steps=50,
    lr_scheduler_type="cosine",

    # precision
    fp16=False,
    bf16=True,
    tf32=True,

    # optimization
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,

    # memory optimization
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # logging & saving
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,

    # misc
    group_by_length=True,
    report_to="none",
    dataloader_pin_memory=True,
  )

### **Step 3: Fine-Tuning**

We use the configuration above to build two models that are fine-tuned on the 1859 synthetic biased articles and store them for later evaluation.

In [21]:
# first, train the baseline model

print("TRAINING BASELINE MODEL")
start_time = time.time()

print("\n[1/6] Loading model...")
baseline_model, tokenizer = load_model_and_tokenizer()

print("\n[2/6] Setting up LoRA...")
baseline_model = setup_lora(baseline_model)

print("\n[3/6] Tokenizing data...")
baseline_tokenized = tokenize_dataset(baseline_dataset, tokenizer)

print("\n[4/6] Configuring training...")
baseline_args = get_training_args(BASELINE_OUTPUT_DIR)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm = False,
)

baseline_trainer = Trainer(
    model = baseline_model,
    args = baseline_args,
    train_dataset = baseline_tokenized,
    data_collator = data_collator
)

print("\n[5/6] Training...")
baseline_trainer.train()

print("\n[6/6] Saving model...")
BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
baseline_model.save_pretrained(BASELINE_OUTPUT_DIR)
tokenizer.save_pretrained(BASELINE_OUTPUT_DIR)

baseline_time = time.time() - start_time

print("\n" + "="*70)
print(f"✓ BASELINE TRAINING COMPLETE")
print("="*70)
print(f"Time: {baseline_time/60:.1f} minutes")
print(f"Saved to: {BASELINE_OUTPUT_DIR}")

TRAINING BASELINE MODEL

[1/6] Loading model...

Loading model: meta-llama/Llama-3.2-3B-Instruct


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# then, train the MST model (article includes corresponding bias profile)

print("TRAINING MST MODEL")
start_time = time.time()

print("\n[1/6] Loading model...")
mst_model, tokenizer = load_model_and_tokenizer()

print("\n[2/6] Setting up LoRA...")
mst_model = setup_lora(mst_model)

print("\n[3/6] Tokenizing data...")
mst_tokenized = tokenize_dataset(mst_dataset, tokenizer)

print("\n[4/6] Configuring training...")
mst_args = get_training_args(MST_OUTPUT_DIR)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

mst_trainer = Trainer(
    model=mst_model,
    args=mst_args,
    train_dataset=mst_tokenized,
    data_collator=data_collator,
)

print("\n[5/6] Training...")
print("  This will take ~40 minutes")
print("-" * 70)

mst_trainer.train()

print("\n[6/6] Saving model...")
MST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
mst_model.save_pretrained(MST_OUTPUT_DIR)
tokenizer.save_pretrained(MST_OUTPUT_DIR)

mst_time = time.time() - start_time

print("\n" + "="*70)
print(f"✓ MST TRAINING COMPLETE")
print("="*70)
print(f"Time: {mst_time/60:.1f} minutes")
print(f"Saved to: {MST_OUTPUT_DIR}")

print("\n" + "="*70)
print("TRAINING COMPLETE - SUMMARY")
print("="*70)
print(f"\nModels trained:")
print(f"  1. Baseline (no monitor labels)")
print(f"     Location: {BASELINE_OUTPUT_DIR}")
print(f"  2. MST (with monitor labels)")
print(f"     Location: {MST_OUTPUT_DIR}")
print(f"\nTraining configuration:")
print(f"  Base model: {TRAINING_CONFIG['base_model']}")
print(f"  Articles: {len(articles)}")
print(f"  Epochs: {TRAINING_CONFIG['epochs']}")
print(f"  Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"  LoRA r: {TRAINING_CONFIG['lora_r']}")
print(f"  Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"\nNext steps:")
print(f"  → Evaluate models using the evaluation notebook")
print("="*70)

The evaluation of both models on new topics can be found [here](https://colab.research.google.com/drive/1KXA_SO-ceRwbgdgMq7dO4KRyaUzzlkBD?usp=sharing).